# GeoNode and GeoServer Integration Tests

## Parameters
This cell is tagged with `parameters`. Papermill will inject values here when the notebook is executed via Kestra.

In [ ]:
# Default parameters (will be overridden by Papermill)
# Security Note: In production, use environment variables or secure credential management
import os
from typing import Dict, Any, Optional

GEONODE_URL = os.getenv("GEONODE_URL", "http://localhost:8000")
GEOSERVER_URL = os.getenv("GEOSERVER_URL", "http://localhost:8080/geoserver")
USERNAME = os.getenv("GEONODE_USERNAME", "admin")
PASSWORD = os.getenv("GEONODE_PASSWORD", "geoserver")
SAMPLE_SHAPEFILE_PATH = os.getenv("SAMPLE_SHAPEFILE_PATH", "../data/sample_vector.shp")
SAMPLE_RASTER_PATH = os.getenv("SAMPLE_RASTER_PATH", "../data/sample_raster.tif")

# Configuration constants
DEFAULT_TIMEOUT = 30  # seconds
UPLOAD_TIMEOUT = 180  # seconds for file uploads
MAX_RETRY_ATTEMPTS = 3
DEFAULT_WORKSPACE = "geonode"

# Variables to store test results
test_results: Dict[str, Dict[str, Any]] = {}

## Library Imports

In [ ]:
import requests
import os
import json
import logging
import time
import re
import xml.etree.ElementTree as ET
from typing import Dict, Any, Optional, Tuple, List
from urllib.parse import urljoin, urlparse
from requests.auth import HTTPBasicAuth
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import warnings

# Optional imports with fallbacks
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    warnings.warn("GeoPandas not available. Some advanced features may be limited.")

try:
    from owslib.wms import WebMapService
    from owslib.wfs import WebFeatureService
    HAS_OWSLIB = True
except ImportError:
    HAS_OWSLIB = False
    warnings.warn("OWSLib not available. Some OGC service tests may be limited.")

# Configure logging with better formatting
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Suppress urllib3 warnings for unverified HTTPS requests
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

## Helper Functions
Common utility functions for interacting with GeoNode/GeoServer.

In [ ]:
# --- Helper Functions ---
def get_geonode_session(geonode_url, username, password):
    """Establishes an authenticated session with GeoNode."""
    session = requests.Session()
    login_url = f"{geonode_url.rstrip('/')}/account/login/"
    
    # First, get the CSRF token
    try:
        response = session.get(login_url, timeout=10)
        response.raise_for_status()
        csrf_token = session.cookies.get('csrftoken', None)
        if not csrf_token:
            # Try to get it from the HTML form if not in cookies yet
            # This is a simplified way; a more robust parser might be needed
            import re
            match = re.search(r'name=["\']csrfmiddlewaretoken["\'] value=["\'](.*?)["\']', response.text)
            if match:
                csrf_token = match.group(1)
        if not csrf_token:
            logger.error("Could not retrieve CSRF token from GeoNode login page.")
            return None
        logger.info("Successfully retrieved CSRF token.")
    except requests.exceptions.RequestException as e:
        logger.error(f"Error accessing GeoNode login page to get CSRF token: {e}")
        return None

    login_data = {
        'username': username,
        'password': password,
        'csrfmiddlewaretoken': csrf_token,
        'next': '/account/profile/' # Redirect to profile page after login
    }
    
    headers = {
        'Referer': login_url
    }
    
    try:
        response = session.post(login_url, data=login_data, headers=headers, timeout=10)
        response.raise_for_status()
        
        if response.url.endswith('/account/login/'):
             # Check for login error messages if redirected back to login
            if "Please enter a correct username and password" in response.text:
                logger.error("GeoNode login failed: Invalid credentials.")
            else:
                logger.error(f"GeoNode login failed. Status: {response.status_code}. Response URL: {response.url}")
            return None
        
        logger.info(f"Successfully logged into GeoNode: {geonode_url}")
        return session
    except requests.exceptions.RequestException as e:
        logger.error(f"GeoNode login request failed: {e}")
        return None

def get_geoserver_auth():
    """Returns the authentication object for GeoServer requests."""
    return HTTPBasicAuth(USERNAME, PASSWORD)

def check_geoserver_status(geoserver_url):
    """Checks if GeoServer is running."""
    try:
        response = requests.get(f"{geoserver_url.rstrip('/')}/rest/workspaces.json", auth=get_geoserver_auth(), timeout=10)
        if response.status_code == 200:
            logger.info(f"GeoServer is accessible at {geoserver_url}")
            return True
        else:
            logger.error(f"GeoServer status check failed. Status: {response.status_code}, Response: {response.text}")
            return False
    except requests.exceptions.RequestException as e:
        logger.error(f"Could not connect to GeoServer at {geoserver_url}: {e}")
        return False

# Example usage of helper (will be part of actual tests later)
# geonode_session = get_geonode_session(GEONODE_URL, USERNAME, PASSWORD)
# if geonode_session:
#     logger.info("GeoNode session obtained.")
# if check_geoserver_status(GEOSERVER_URL):
#     logger.info("GeoServer status OK.")

# Add a test result utility
def record_test_result(test_name, status, message=""):
    test_results[test_name] = {"status": "SUCCESS" if status else "FAILURE", "message": message}
    if status:
        logger.info(f"Test '{test_name}': SUCCESS. {message}")
    else:
        logger.error(f"Test '{test_name}': FAILURE. {message}")

# --- End Helper Functions ---

## Test Case 1: Load Data
Upload sample vector and raster data.

In [ ]:
# --- Test Case 1: Load Data ---

# Ensure GEONODE_URL, USERNAME, PASSWORD, SAMPLE_SHAPEFILE_PATH, SAMPLE_RASTER_PATH are defined (from parameters cell)
# Ensure geonode_session is established, and test_results, record_test_result are available

# Initialize GeoNode session for this test block
geonode_session = get_geonode_session(GEONODE_URL, USERNAME, PASSWORD)
if not geonode_session:
    record_test_result("geonode_login_for_data_upload", False, "Failed to log into GeoNode. Skipping data uploads.")
else:
    logger.info("Successfully logged into GeoNode for data upload.")

    # --- Vector Data Upload (Shapefile) ---
    def upload_shapefile_to_geonode(session, geonode_url, file_path, permissions=None):
        """
        Uploads a shapefile (as a zip or individual files) to GeoNode.
        GeoNode's /layers/upload endpoint expects a multipart form.
        It's often easier to zip the shapefile components first.
        For this example, we'll assume direct upload of constituent files if unzipped,
        or a single .zip if `file_path` points to a zip.
        Adjust based on actual GeoNode API behavior for direct file array uploads.
        
        Note: GeoNode's API for file uploads can be complex.
        It might require specific naming for form fields or additional metadata.
        This is a simplified example. A library like 'geonode-py' would abstract this.
        We will assume the API can handle the base file and will discover related files.
        """
        test_name = f"upload_shapefile_{os.path.basename(file_path)}"
        if not os.path.exists(file_path):
            record_test_result(test_name, False, f"Shapefile not found at {file_path}")
            return None

        upload_url = f"{geonode_url.rstrip('/')}/layers/upload"
        
        files_to_upload = {}
        base, ext = os.path.splitext(file_path)
        if ext.lower() == '.shp':
            # Try to gather all parts of the shapefile
            shapefile_parts = [
                file_path, f"{base}.shx", f"{base}.dbf", f"{base}.prj", 
                f"{base}.sld", f"{base}.qml" # Optional styling files
            ]
            actual_files_found = False
            for part_path in shapefile_parts:
                if os.path.exists(part_path):
                    files_to_upload[os.path.basename(part_path)] = (os.path.basename(part_path), open(part_path, 'rb'))
                    actual_files_found = True
            if not actual_files_found:
                record_test_result(test_name, False, f"No valid shapefile parts found for base {base}")
                return None
        else:
            # Assuming it's a zip or other single file format GeoNode supports for vector layers
            files_to_upload['file'] = (os.path.basename(file_path), open(file_path, 'rb'))

        # GeoNode might require a CSRF token even for API uploads if session based
        # The session object should handle cookies, including CSRF if set during login.
        # However, some API endpoints might need it explicitly in headers or form data.
        # Let's try without first, assuming session cookies are sufficient.
        
        # Default permissions: anyone can view, only owner can edit
        default_permissions = {
            "users": {"AnonymousUser": ["view_resourcebase"]},
            "groups": {}
        }
        form_data = {
            'permissions': json.dumps(permissions or default_permissions),
            # 'store_type': 'dataStore', # May not be needed, GeoNode might infer
            # 'override_existing_layer': True, # Optional
        }

        try:
            # The 'X-CSRFToken' header might be needed if cookies aren't automatically used by the endpoint
            csrf_token = session.cookies.get('csrftoken')
            headers = {}
            if csrf_token:
                headers['X-CSRFToken'] = csrf_token
            
            logger.info(f"Attempting to upload {file_path} to {upload_url}")
            response = session.post(upload_url, files=files_to_upload, data=form_data, headers=headers, timeout=120) # Increased timeout for uploads
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

            # Response handling can be tricky. GeoNode might return a JSON with status or redirect.
            # A 200 or 201 or 202 is usually good.
            # If it redirects, the final URL might indicate success (e.g., to the layer's page).
            if response.status_code in [200, 201, 202]:
                logger.info(f"Upload request for {file_path} successful. Status: {response.status_code}. Response: {response.text[:200]}")
                # The response might contain the new layer name or URL.
                # For now, we just confirm the upload request was accepted.
                # A more robust check would be to query GeoNode for the layer after upload.
                layer_name = os.path.splitext(os.path.basename(file_path))[0] # Guess layer name
                record_test_result(test_name, True, f"Successfully initiated upload. Layer name (guessed): {layer_name}")
                return layer_name # Or some identifier from response
            else:
                record_test_result(test_name, False, f"Upload failed. Status: {response.status_code}, Response: {response.text}")
                return None

        except requests.exceptions.HTTPError as e:
            record_test_result(test_name, False, f"HTTP error during upload: {e}. Response: {e.response.text if e.response else 'No response body'}")
            return None
        except requests.exceptions.RequestException as e:
            record_test_result(test_name, False, f"Error during upload: {e}")
            return None
        finally:
            for f_info in files_to_upload.values():
                f_info[1].close()


    # --- Raster Data Upload (GeoTIFF) ---
    def upload_geotiff_to_geonode(session, geonode_url, file_path, permissions=None):
        """Uploads a GeoTIFF file to GeoNode."""
        test_name = f"upload_geotiff_{os.path.basename(file_path)}"
        if not os.path.exists(file_path):
            record_test_result(test_name, False, f"GeoTIFF not found at {file_path}")
            return None

        upload_url = f"{geonode_url.rstrip('/')}/layers/upload" # Same endpoint often handles various types
        
        files = {'base_file': (os.path.basename(file_path), open(file_path, 'rb'))}
        # For rasters, GeoNode might also need 'metadata_upload_form-store_type': 'coverageStore'
        # and potentially other form fields. This depends on the GeoNode version and configuration.

        default_permissions = {
            "users": {"AnonymousUser": ["view_resourcebase"]},
            "groups": {}
        }
        form_data = {
            'permissions': json.dumps(permissions or default_permissions),
            # 'store_type': 'coverageStore', # May not be needed
        }

        try:
            csrf_token = session.cookies.get('csrftoken')
            headers = {}
            if csrf_token:
                headers['X-CSRFToken'] = csrf_token

            logger.info(f"Attempting to upload {file_path} to {upload_url}")
            response = session.post(upload_url, files=files, data=form_data, headers=headers, timeout=180) # Increased timeout
            response.raise_for_status()

            if response.status_code in [200, 201, 202]:
                logger.info(f"Upload request for {file_path} successful. Status: {response.status_code}. Response: {response.text[:200]}")
                layer_name = os.path.splitext(os.path.basename(file_path))[0] # Guess layer name
                record_test_result(test_name, True, f"Successfully initiated upload. Layer name (guessed): {layer_name}")
                return layer_name
            else:
                record_test_result(test_name, False, f"Upload failed. Status: {response.status_code}, Response: {response.text}")
                return None
        except requests.exceptions.HTTPError as e:
            record_test_result(test_name, False, f"HTTP error during upload: {e}. Response: {e.response.text if e.response else 'No response body'}")
            return None
        except requests.exceptions.RequestException as e:
            record_test_result(test_name, False, f"Error during upload: {e}")
            return None
        finally:
            if 'base_file' in files:
                files['base_file'][1].close()

    # Execute data loading tests if GeoNode session is available
    if geonode_session:
        # Define expected layer names (these should match the filenames without extension)
        # These will be used in later tests.
        # It's important these are consistent.
        EXPECTED_VECTOR_LAYER_NAME = os.path.splitext(os.path.basename(SAMPLE_SHAPEFILE_PATH))[0]
        EXPECTED_RASTER_LAYER_NAME = os.path.splitext(os.path.basename(SAMPLE_RASTER_PATH))[0]

        logger.info(f"Expected vector layer name: {EXPECTED_VECTOR_LAYER_NAME}")
        logger.info(f"Expected raster layer name: {EXPECTED_RASTER_LAYER_NAME}")

        # Upload Shapefile
        # Note: GeoNode might create the layer with a modified name (e.g., if it already exists or due to sanitization).
        # The functions above return the guessed name, but a more robust approach would be to
        # get the actual layer name from GeoNode's response if available, or query by filename.
        uploaded_vector_layer_name = upload_shapefile_to_geonode(geonode_session, GEONODE_URL, SAMPLE_SHAPEFILE_PATH)
        if uploaded_vector_layer_name:
            # For subsequent tests, we should use the *actual* name GeoNode assigns.
            # For now, we assume the guessed name is correct or close enough.
            # If GeoNode uses workspace:name format, that needs to be handled.
            # We might need a function like `get_layer_details_from_geonode(layer_name)`
            logger.info(f"Shapefile upload process initiated. Assumed layer name: {uploaded_vector_layer_name}")
        else:
            logger.error("Shapefile upload process failed.")

        # Upload GeoTIFF
        uploaded_raster_layer_name = upload_geotiff_to_geonode(geonode_session, GEONODE_URL, SAMPLE_RASTER_PATH)
        if uploaded_raster_layer_name:
            logger.info(f"GeoTIFF upload process initiated. Assumed layer name: {uploaded_raster_layer_name}")
        else:
            logger.error("GeoTIFF upload process failed.")
    else:
        # Record failures if login itself failed
        record_test_result(f"upload_shapefile_{os.path.basename(SAMPLE_SHAPEFILE_PATH)}", False, "Skipped due to GeoNode login failure.")
        record_test_result(f"upload_geotiff_{os.path.basename(SAMPLE_RASTER_PATH)}", False, "Skipped due to GeoNode login failure.")


# --- End Test Case 1 ---

## Test Case 2: Create Data Store (Vector)
If not automatically handled by GeoNode, explicitly create a vector data store.

In [ ]:
# --- Test Case 2: Verify Vector Data Store in GeoServer ---

# Assumes GEOSERVER_URL, USERNAME, PASSWORD are defined.
# Assumes EXPECTED_VECTOR_LAYER_NAME is defined from Test Case 1.
# Uses get_geoserver_auth() and record_test_result() from helper functions.

def check_geoserver_datastore(geoserver_url, auth, workspace_name, datastore_name):
    """Checks if a specific datastore exists in a GeoServer workspace."""
    # GeoServer often creates datastores with the same name as the layer.
    # If GeoNode uses a specific workspace, it should be part of the layer name or known.
    # For now, assume the layer name is the datastore name, and it's in the default GeoNode workspace (e.g., 'geonode')
    # or the global scope if no specific workspace is used by GeoNode for publishing.

    # If EXPECTED_VECTOR_LAYER_NAME contains a colon, it implies workspace:layer
    if ':' in datastore_name:
        ws_name, ds_name = datastore_name.split(':', 1)
    else:
        # If no workspace in layer name, use the provided workspace_name parameter
        # This might be 'geonode' or some other default.
        # Or, if GeoNode layers are published without a workspace prefix in their name,
        # they might be in a default workspace like 'cite' or 'gs'.
        # This part is highly dependent on the GeoNode <-> GeoServer configuration.
        # Let's assume for now workspace_name parameter is the one to check.
        ws_name = workspace_name
        ds_name = datastore_name

    url = f"{geoserver_url.rstrip('/')}/rest/workspaces/{ws_name}/datastores/{ds_name}.json"
    test_name = f"check_datastore_{ws_name}_{ds_name}"
    
    try:
        logger.info(f"Checking for datastore: {ws_name}:{ds_name} at {url}")
        response = requests.get(url, auth=auth, timeout=10)
        if response.status_code == 200:
            record_test_result(test_name, True, f"Data store '{ws_name}:{ds_name}' found in GeoServer.")
            return True
        elif response.status_code == 404:
            record_test_result(test_name, False, f"Data store '{ws_name}:{ds_name}' not found in GeoServer.")
            return False
        else:
            record_test_result(test_name, False, f"Error checking data store '{ws_name}:{ds_name}'. Status: {response.status_code}, Response: {response.text}")
            return False
    except requests.exceptions.RequestException as e:
        record_test_result(test_name, False, f"RequestException while checking data store '{ws_name}:{ds_name}': {e}")
        return False

# Determine the workspace and datastore name to check.
# This often depends on how GeoNode is configured to publish layers to GeoServer.
# Common GeoNode workspace is 'geonode'. If layer names from GeoNode are like 'layer1',
# then in GeoServer it might be 'geonode:layer1'.
# If EXPECTED_VECTOR_LAYER_NAME is already 'geonode:sample_vector', then it's handled.
# For this test, we'll assume a default workspace 'geonode' if not part of the name.
GS_WORKSPACE_NAME = "geonode" # This might need to be a parameter or discovered

# Check if GeoServer is running before proceeding
if check_geoserver_status(GEOSERVER_URL):
    gs_auth = get_geoserver_auth()
    
    # Check for the vector datastore
    # We use EXPECTED_VECTOR_LAYER_NAME which should be defined in Test Case 1
    # It's assumed that GeoNode creates a datastore with a name matching the layer name.
    if 'EXPECTED_VECTOR_LAYER_NAME' in locals() or 'EXPECTED_VECTOR_LAYER_NAME' in globals():
        vector_ds_name_to_check = EXPECTED_VECTOR_LAYER_NAME
        logger.info(f"Proceeding to check vector datastore: {vector_ds_name_to_check} in workspace {GS_WORKSPACE_NAME}")
        check_geoserver_datastore(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_ds_name_to_check)
    else:
        record_test_result(f"check_datastore_vector_skipped", False, "Skipped: EXPECTED_VECTOR_LAYER_NAME not defined from data upload step.")
else:
    record_test_result("check_datastore_vector_skipped_geoserver_down", False, "Skipped: GeoServer is not accessible.")

# --- End Test Case 2 ---

## Test Case 3: Create Coverage Store (Raster)
If not automatically handled by GeoNode, explicitly create a coverage store.

In [ ]:
# --- Test Case 3: Verify Coverage Store in GeoServer ---

# Assumes GEOSERVER_URL, USERNAME, PASSWORD are defined.
# Assumes EXPECTED_RASTER_LAYER_NAME is defined from Test Case 1.
# Uses get_geoserver_auth() and record_test_result() from helper functions.

def check_geoserver_coveragestore(geoserver_url, auth, workspace_name, coveragestore_name):
    """Checks if a specific coverage store exists in a GeoServer workspace."""
    # Similar logic to datastore checking for workspace and store name.
    if ':' in coveragestore_name:
        ws_name, cs_name = coveragestore_name.split(':', 1)
    else:
        ws_name = workspace_name
        cs_name = coveragestore_name
        
    url = f"{geoserver_url.rstrip('/')}/rest/workspaces/{ws_name}/coveragestores/{cs_name}.json"
    test_name = f"check_coveragestore_{ws_name}_{cs_name}"

    try:
        logger.info(f"Checking for coveragestore: {ws_name}:{cs_name} at {url}")
        response = requests.get(url, auth=auth, timeout=10)
        if response.status_code == 200:
            record_test_result(test_name, True, f"Coverage store '{ws_name}:{cs_name}' found in GeoServer.")
            return True
        elif response.status_code == 404:
            record_test_result(test_name, False, f"Coverage store '{ws_name}:{cs_name}' not found in GeoServer.")
            return False
        else:
            record_test_result(test_name, False, f"Error checking coverage store '{ws_name}:{cs_name}'. Status: {response.status_code}, Response: {response.text}")
            return False
    except requests.exceptions.RequestException as e:
        record_test_result(test_name, False, f"RequestException while checking coverage store '{ws_name}:{cs_name}': {e}")
        return False

# Check if GeoServer is running before proceeding
if check_geoserver_status(GEOSERVER_URL):
    gs_auth = get_geoserver_auth() # Re-auth just in case, though session might persist

    # Check for the raster coverage store
    # We use EXPECTED_RASTER_LAYER_NAME which should be defined in Test Case 1
    # Assumed that GeoNode creates a coverage store with a name matching the layer name.
    if 'EXPECTED_RASTER_LAYER_NAME' in locals() or 'EXPECTED_RASTER_LAYER_NAME' in globals():
        raster_cs_name_to_check = EXPECTED_RASTER_LAYER_NAME
        logger.info(f"Proceeding to check raster coveragestore: {raster_cs_name_to_check} in workspace {GS_WORKSPACE_NAME}")
        check_geoserver_coveragestore(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, raster_cs_name_to_check)
    else:
        record_test_result("check_coveragestore_raster_skipped", False, "Skipped: EXPECTED_RASTER_LAYER_NAME not defined from data upload step.")
else:
    # This message might be redundant if already recorded by datastore check, but good for separation
    record_test_result("check_coveragestore_raster_skipped_geoserver_down", False, "Skipped: GeoServer is not accessible.")

# --- End Test Case 3 ---

## Test Case 4: Verify Layer Publishing
Check if layers are published as WMS and WFS.

In [ ]:
# --- Test Case 4: Verify Layer Publishing in GeoServer ---

# Assumes GEOSERVER_URL, USERNAME, PASSWORD are defined.
# Assumes EXPECTED_VECTOR_LAYER_NAME and EXPECTED_RASTER_LAYER_NAME are defined from Test Case 1.
# Uses get_geoserver_auth(), record_test_result(), and GS_WORKSPACE_NAME.

def check_geoserver_layer(geoserver_url, auth, workspace_name, layer_name):
    """
    Checks if a specific layer exists in GeoServer and is enabled.
    The layer_name here should be the plain name, not workspace:name.
    The workspace_name parameter specifies the workspace.
    """
    # Construct the full layer name for messages, GeoServer sometimes uses this in responses
    qualified_layer_name = f"{workspace_name}:{layer_name}"
    
    # URL to get layer details from GeoServer
    # Note: GeoServer's layer names might or might not include the workspace prefix depending on the endpoint.
    # /rest/layers/{layerName}.json usually expects the simple layer name if it's not in default workspace.
    # /rest/workspaces/{workspaceName}/layers/{layerName}.json is more specific.
    # Let's try to be specific with workspace.
    # However, GeoNode often creates layers whose names *already include* the workspace prefix,
    # e.g. layer name in GeoNode is 'mylayer', but in GeoServer it's 'geonode:mylayer'.
    # So, if layer_name already contains ':', we should use that.
    
    actual_layer_name_to_check = layer_name
    actual_workspace_name = workspace_name

    if ':' in layer_name:
        # If layer_name is like 'geonode:sample_vector', then split it
        ws_from_layer, ln_from_layer = layer_name.split(':', 1)
        actual_workspace_name = ws_from_layer
        actual_layer_name_to_check = ln_from_layer
        qualified_layer_name = layer_name # Already qualified
        
    url = f"{geoserver_url.rstrip('/')}/rest/workspaces/{actual_workspace_name}/layers/{actual_layer_name_to_check}.json"
    test_name = f"check_layer_published_{qualified_layer_name}"

    try:
        logger.info(f"Checking for layer: {qualified_layer_name} via URL: {url}")
        response = requests.get(url, auth=auth, headers={'Accept': 'application/json'}, timeout=10)
        
        if response.status_code == 200:
            layer_info = response.json().get('layer', {})
            if layer_info.get('enabled'):
                # Also check if it's advertised, though 'enabled' is a primary check
                advertised = layer_info.get('advertised', True) # Default to True if key missing but enabled
                if advertised:
                    record_test_result(test_name, True, f"Layer '{qualified_layer_name}' found, enabled, and advertised in GeoServer.")
                    return True
                else:
                    record_test_result(test_name, False, f"Layer '{qualified_layer_name}' found and enabled, but NOT advertised in GeoServer.")
                    return False
            else:
                record_test_result(test_name, False, f"Layer '{qualified_layer_name}' found but NOT enabled in GeoServer.")
                return False
        elif response.status_code == 404:
            record_test_result(test_name, False, f"Layer '{qualified_layer_name}' not found in GeoServer at workspace '{actual_workspace_name}'.")
            return False
        else:
            record_test_result(test_name, False, f"Error checking layer '{qualified_layer_name}'. Status: {response.status_code}, Response: {response.text}")
            return False
    except requests.exceptions.RequestException as e:
        record_test_result(test_name, False, f"RequestException while checking layer '{qualified_layer_name}': {e}")
        return False
    except json.JSONDecodeError as e:
        record_test_result(test_name, False, f"JSONDecodeError while checking layer '{qualified_layer_name}': {e}. Response text: {response.text}")
        return False


# Check if GeoServer is running before proceeding
if check_geoserver_status(GEOSERVER_URL):
    gs_auth = get_geoserver_auth()

    # Check for the published vector layer
    if 'EXPECTED_VECTOR_LAYER_NAME' in locals() or 'EXPECTED_VECTOR_LAYER_NAME' in globals():
        vector_layer_name_to_check = EXPECTED_VECTOR_LAYER_NAME
        logger.info(f"Proceeding to check published vector layer: {vector_layer_name_to_check} in workspace {GS_WORKSPACE_NAME}")
        check_geoserver_layer(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_layer_name_to_check)
    else:
        record_test_result("check_layer_published_vector_skipped", False, "Skipped: EXPECTED_VECTOR_LAYER_NAME not defined.")

    # Check for the published raster layer
    if 'EXPECTED_RASTER_LAYER_NAME' in locals() or 'EXPECTED_RASTER_LAYER_NAME' in globals():
        raster_layer_name_to_check = EXPECTED_RASTER_LAYER_NAME
        logger.info(f"Proceeding to check published raster layer: {raster_layer_name_to_check} in workspace {GS_WORKSPACE_NAME}")
        check_geoserver_layer(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, raster_layer_name_to_check)
    else:
        record_test_result("check_layer_published_raster_skipped", False, "Skipped: EXPECTED_RASTER_LAYER_NAME not defined.")
else:
    record_test_result("check_layer_published_skipped_geoserver_down", False, "Skipped layer publishing checks: GeoServer is not accessible.")

# --- End Test Case 4 ---

## Test Case 5: Test WMS Service
Send GetMap requests and verify responses.

In [ ]:
# --- Test Case 5: Test WMS Service ---

# Assumes GEOSERVER_URL, USERNAME, PASSWORD are defined.
# Assumes EXPECTED_VECTOR_LAYER_NAME and EXPECTED_RASTER_LAYER_NAME are defined.
# Uses get_geoserver_auth(), record_test_result(), and GS_WORKSPACE_NAME.

def test_wms_getmap(geoserver_url, auth, workspace_name, layer_name, bbox=None, image_format="image/png"):
    """
    Tests the WMS GetMap request for a given layer.
    The layer_name can be 'workspace:layer' or just 'layer' if workspace_name is provided separately.
    """
    
    # Construct the qualified layer name for the WMS request
    # WMS typically expects layers in 'workspace:layer' format if they are not in the default workspace.
    qualified_wms_layer_name = layer_name
    if ':' not in layer_name:
        qualified_wms_layer_name = f"{workspace_name}:{layer_name}"

    test_name = f"wms_getmap_{qualified_wms_layer_name.replace(':', '_')}" # Sanitize for test name

    # Default BBOX (replace with actual layer bbox if discoverable and needed)
    # These are generic WGS84 bounds, may not be appropriate for all layers.
    # A better approach would be to get layer's native bbox from capabilities or GeoServer REST API.
    default_bbox = "-180,-90,180,90" # General WGS84 extent
    if layer_name == EXPECTED_RASTER_LAYER_NAME: # Example: Raster might have specific extent
        # This should ideally come from layer metadata
        default_bbox = "0,0,10,10" # Placeholder, replace with actual if known
    
    current_bbox = bbox if bbox else default_bbox

    # Construct WMS GetMap URL
    wms_base_url = f"{geoserver_url.rstrip('/')}/{workspace_name}/wms" # Or just /geoserver/wms if workspace is in layer name
    # If layer name already includes workspace, the WMS endpoint might just be /geoserver/wms
    if ':' in layer_name : # e.g. layer_name is 'geonode:mylayer'
         wms_base_url = f"{geoserver_url.rstrip('/')}/wms"


    params = {
        'SERVICE': 'WMS',
        'VERSION': '1.1.1', # Or 1.3.0
        'REQUEST': 'GetMap',
        'LAYERS': qualified_wms_layer_name,
        'STYLES': '',
        'SRS': 'EPSG:4326', # Or other appropriate CRS for the layer
        'BBOX': current_bbox,
        'WIDTH': '768',
        'HEIGHT': '384',
        'FORMAT': image_format
    }

    try:
        logger.info(f"Requesting WMS GetMap for layer '{qualified_wms_layer_name}' with BBOX: {current_bbox}")
        # For WMS, auth might be part of query params for some servers, but HTTPBasicAuth is standard
        response = requests.get(wms_base_url, params=params, auth=auth, timeout=20)

        if response.status_code == 200:
            content_type = response.headers.get('Content-Type', '').lower()
            if image_format.lower() in content_type:
                # Basic check for image content (non-empty)
                if len(response.content) > 100: # Arbitrary small size to indicate some image data
                    record_test_result(test_name, True, f"WMS GetMap for '{qualified_wms_layer_name}' returned 200 OK with correct Content-Type '{content_type}' and non-empty content.")
                    return True
                else:
                    record_test_result(test_name, False, f"WMS GetMap for '{qualified_wms_layer_name}' returned 200 OK with '{content_type}', but content is too small (possibly empty image).")
                    return False
            else:
                record_test_result(test_name, False, f"WMS GetMap for '{qualified_wms_layer_name}' returned 200 OK, but incorrect Content-Type. Expected '{image_format}', got '{content_type}'. Response text (partial): {response.text[:200]}")
                # This could be an XML exception from GeoServer
                if "ServiceException" in response.text:
                     logger.error(f"GeoServer WMS ServiceException: {response.text}")
                return False
        else:
            record_test_result(test_name, False, f"WMS GetMap for '{qualified_wms_layer_name}' failed. Status: {response.status_code}, Response: {response.text[:500]}")
            return False
            
    except requests.exceptions.RequestException as e:
        record_test_result(test_name, False, f"RequestException during WMS GetMap for '{qualified_wms_layer_name}': {e}")
        return False

# Check if GeoServer is running before proceeding
if check_geoserver_status(GEOSERVER_URL):
    gs_auth = get_geoserver_auth()

    # Test WMS for the vector layer
    if 'EXPECTED_VECTOR_LAYER_NAME' in locals() or 'EXPECTED_VECTOR_LAYER_NAME' in globals():
        vector_layer_wms_name = EXPECTED_VECTOR_LAYER_NAME
        # Bounding box for vector layer - this should ideally be fetched from the layer's metadata
        # For now, using a generic one. If the layer is in 'geonode' workspace, name is 'geonode:sample_vector'
        # The test_wms_getmap function handles prepending GS_WORKSPACE_NAME if not already part of vector_layer_wms_name
        logger.info(f"Proceeding to test WMS GetMap for vector layer: {vector_layer_wms_name}")
        test_wms_getmap(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_layer_wms_name)
    else:
        record_test_result("wms_getmap_vector_skipped", False, "Skipped: EXPECTED_VECTOR_LAYER_NAME not defined.")

    # Test WMS for the raster layer
    if 'EXPECTED_RASTER_LAYER_NAME' in locals() or 'EXPECTED_RASTER_LAYER_NAME' in globals():
        raster_layer_wms_name = EXPECTED_RASTER_LAYER_NAME
        # Bounding box for raster - this should ideally be fetched.
        # Using a generic one for now.
        logger.info(f"Proceeding to test WMS GetMap for raster layer: {raster_layer_wms_name}")
        test_wms_getmap(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, raster_layer_wms_name)
    else:
        record_test_result("wms_getmap_raster_skipped", False, "Skipped: EXPECTED_RASTER_LAYER_NAME not defined.")
else:
    record_test_result("wms_getmap_skipped_geoserver_down", False, "Skipped WMS tests: GeoServer is not accessible.")

# --- End Test Case 5 ---

## Test Case 6: Test WFS Service
Send GetFeature requests (GML, GeoJSON, JSON) and verify responses.

In [ ]:
# --- Test Case 6: Test WFS Service ---

# Assumes GEOSERVER_URL, USERNAME, PASSWORD are defined.
# Assumes EXPECTED_VECTOR_LAYER_NAME is defined.
# Uses get_geoserver_auth(), record_test_result(), and GS_WORKSPACE_NAME.
# May need xml.etree.ElementTree for GML parsing if not using owslib.

import xml.etree.ElementTree as ET

def test_wfs_getfeature(geoserver_url, auth, workspace_name, layer_name, output_format="application/json", max_features=1):
    """
    Tests the WFS GetFeature request for a given layer and output format.
    Layer_name can be 'workspace:layer' or just 'layer'.
    """
    qualified_wfs_layer_name = layer_name
    if ':' not in layer_name:
        qualified_wfs_layer_name = f"{workspace_name}:{layer_name}"

    # Sanitize format for test name (e.g., application/json -> application_json)
    format_suffix = output_format.split('/')[-1].replace('+', '_').replace(';', '_')
    test_name = f"wfs_getfeature_{qualified_wfs_layer_name.replace(':', '_')}_{format_suffix}"

    wfs_base_url = f"{geoserver_url.rstrip('/')}/{workspace_name}/wfs" # Or just /geoserver/wfs
    if ':' in layer_name: # e.g. layer_name is 'geonode:mylayer'
        wfs_base_url = f"{geoserver_url.rstrip('/')}/wfs"

    params = {
        'SERVICE': 'WFS',
        'VERSION': '1.1.0', # Or 1.0.0 / 2.0.0
        'REQUEST': 'GetFeature',
        'TYPENAME': qualified_wfs_layer_name,
        'OUTPUTFORMAT': output_format,
        'MAXFEATURES': str(max_features) # Limit number of features for test
    }

    try:
        logger.info(f"Requesting WFS GetFeature for layer '{qualified_wfs_layer_name}' in format '{output_format}'")
        response = requests.get(wfs_base_url, params=params, auth=auth, timeout=20)

        if response.status_code == 200:
            content_type = response.headers.get('Content-Type', '').lower()
            # GeoServer might return charset, so check if output_format is a substring
            if output_format.lower() in content_type:
                # Basic content validation
                if len(response.content) > 0:
                    valid_content = False
                    if "json" in output_format.lower():
                        try:
                            data = response.json()
                            if 'features' in data and isinstance(data['features'], list):
                                if max_features == 0 or len(data['features']) <= max_features : # if max_features=0, it means any number of features is ok
                                     record_test_result(test_name, True, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, valid JSON with features.")
                                     valid_content = True
                                else:
                                     record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but number of features {len(data['features'])} > max_features {max_features}.")
                            else:
                                record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but JSON does not contain a 'features' list.")
                        except json.JSONDecodeError:
                            record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but content is not valid JSON.")
                    elif "gml" in output_format.lower() or "xml" in output_format.lower():
                        try:
                            # Basic XML validation: try to parse and check root element
                            root = ET.fromstring(response.content)
                            # GML often has a <wfs:FeatureCollection> or similar root, or directly the feature type
                            # For simplicity, just checking it's parsable XML.
                            # A more robust check would involve XML schema validation or checking for specific GML tags.
                            if root is not None:
                                record_test_result(test_name, True, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, valid XML.")
                                valid_content = True
                            else:
                                record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but XML parsing yielded empty root.")
                        except ET.ParseError:
                            record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but content is not valid XML.")
                    else: # Other formats not specifically checked for content structure
                        record_test_result(test_name, True, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK with correct Content-Type and non-empty content.")
                        valid_content = True
                    return valid_content
                else:
                    record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK with '{content_type}', but content is empty.")
                    return False
            else:
                record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) returned 200 OK, but incorrect Content-Type. Expected something like '{output_format}', got '{content_type}'. Response text (partial): {response.text[:200]}")
                if "ServiceException" in response.text:
                     logger.error(f"GeoServer WFS ServiceException: {response.text}")
                return False
        else:
            record_test_result(test_name, False, f"WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}) failed. Status: {response.status_code}, Response: {response.text[:500]}")
            return False

    except requests.exceptions.RequestException as e:
        record_test_result(test_name, False, f"RequestException during WFS GetFeature for '{qualified_wfs_layer_name}' ({output_format}): {e}")
        return False

# Check if GeoServer is running before proceeding
if check_geoserver_status(GEOSERVER_URL):
    gs_auth = get_geoserver_auth()

    # Test WFS for the vector layer (raster layers are not typically served via WFS GetFeature)
    if 'EXPECTED_VECTOR_LAYER_NAME' in locals() or 'EXPECTED_VECTOR_LAYER_NAME' in globals():
        vector_layer_wfs_name = EXPECTED_VECTOR_LAYER_NAME
        logger.info(f"Proceeding to test WFS GetFeature for vector layer: {vector_layer_wfs_name}")

        # Test with GeoJSON (common)
        test_wfs_getfeature(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_layer_wfs_name, output_format="application/json", max_features=1)
        
        # Test with GML (another common format, GeoServer often defaults to a GML variant)
        # Using a common GML type, specific version might vary.
        test_wfs_getfeature(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_layer_wfs_name, output_format="application/gml+xml; version=3.2", max_features=1)
        
        # Test with older GML version (sometimes 'text/xml; subtype=gml/3.1.1' or similar)
        test_wfs_getfeature(GEOSERVER_URL, gs_auth, GS_WORKSPACE_NAME, vector_layer_wfs_name, output_format="text/xml; subtype=gml/2.1.2", max_features=1)


    else:
        record_test_result("wfs_getfeature_vector_skipped", False, "Skipped: EXPECTED_VECTOR_LAYER_NAME not defined.")
else:
    record_test_result("wfs_getfeature_skipped_geoserver_down", False, "Skipped WFS tests: GeoServer is not accessible.")

# --- End Test Case 6 ---

## Test Summary

In [ ]:
# Placeholder for printing test_results
logger.info(f"Test Results: {json.dumps(test_results, indent=2)}")